# Analisis Dataset Diabetes



## 1. Setup & Load Data

Impor library yang dibutuhkan dan muat dataset.

In [2]:
# Setup & load
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import os



 Problem Statement & Tujuan

 **Permasalahan:**
 Diabetes adalah penyakit kronis dengan prevalensi tinggi di Indonesia.
 Deteksi dini diabetes dapat mencegah komplikasi serius seperti gagal ginjal,
 kebutaan, dan penyakit kardiovaskular. Namun, banyak kasus diabetes tidak terdeteksi
 hingga stadium lanjut karena minimnya screening rutin.

 **Tujuan:**
 1. Membangun model prediksi diabetes berdasarkan data medis dasar
 2. Mengidentifikasi faktor risiko utama diabetes
 3. Memberikan rekomendasi screening prioritas untuk populasi berisiko tinggi

 **Dataset:**
 - Sumber: Kaggle Diabetes Prediction Dataset (synthetic medical records)
 - Total: 5288 pasien dengan 13 variabel medis
 - Target: Status diabetes (Yes/No)

In [3]:

df = pd.read_csv("diabetes_dataset_dirty.csv", delimiter=';')
print("Shape:", df.shape)
df.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'diabetes_dataset_dirty.csv'

In [ ]:
print(df['diabetic'].value_counts())

In [ ]:
print("=== INFO DATA ===")
print(df.info())

In [ ]:
print("\n=== CEK MISSING VALUES ===")
print(df.isnull().sum())

In [ ]:
print("\n=== CEK DUPLIKAT ===")
print(f"Jumlah data dupliat:{df.duplicated().sum()}")

In [ ]:
print("\n=== STATISTIK NUMERIK ===")
print(df.describe())

In [ ]:
print("\n=== CEK NILAI UNIK KATEGORIK ===")
for col in df.select_dtypes(include=['object']).columns:
    print(col, df[col].unique())

In [ ]:
# menormalisasikan tinggi badan menjadi ratusan, data ada yg hilang nolnya jadi cuman 15, 16 dll

def fix_height_cm(x):
    if 10 <= x < 100:
        return x * 10
    elif 1 <= x < 10:
        return x * 100
    else:
        return x

df["height"] = df["height"].apply(fix_height_cm)

print(df["height"].describe())


In [ ]:
df.head(30)

In [ ]:
# berat badan kemungkinan besar dalam kg, dan ada koma jadi saya perbaiki

def fix_weight(x):
    if x > 200:
        # Misal 425 → 42.5
        return x / 10
    elif x < 30:
        # Misal 7 → 70
        return x * 10
    else:
        return x

df["weight"] = df["weight"].apply(fix_weight)
print(df["weight"].describe())



In [ ]:
df.head(30)

In [ ]:
# menghitung ulang bmi berdasarkan rumus, koma tidak disederhanakan agar lebih presisi
# Fix height
def fix_height_cm(x):
    if 10 <= x < 100:
        return x * 10
    elif 1 <= x < 10:
        return x * 100
    else:
        return x

df["height"] = df["height"].apply(fix_height_cm)

# Fix weight
def fix_weight(x):
    if x > 200:
        return x / 10
    elif x < 30:
        return x * 10
    else:
        return x

df["weight"] = df["weight"].apply(fix_weight)

# Hitung ulang BMI
df["bmi"] = df["weight"] / (df["height"]/100)**2

print(df[["height","weight","bmi"]])


In [ ]:
df.head(10)

In [ ]:
# normalisasi glucose dan outlier

# Fungsi normalisasi glucose
def normalize_glucose(x):
    """
    Normalisasi glucose:
    - Jika <40 → kemungkinan hilang koma → kali 10
    - Jika >400 → jelas salah input → bagi 10
    - Jika 40-400 → biarkan
    """
    if x < 40:
        return x * 10
    elif x > 400:
        return x / 10
    else:
        return x

# Terapkan normalisasi
df['glucose'] = df['glucose'].apply(normalize_glucose)

# Ganti nilai di bawah 40 atau di atas 400 setelah koreksi (jika masih ada) dengan NaN
df.loc[df['glucose'] < 40, 'glucose'] = np.nan
df.loc[df['glucose'] > 400, 'glucose'] = np.nan

# Ringkasan statistik
print(df['glucose'].describe())
print("\nData setelah normalisasi:")
print(df)

In [ ]:
# mengatasi outlier ekstrim height

# Ganti height > 250 menjadi NaN
df.loc[df['height'] > 250, 'height'] = np.nan

# (Opsional) ganti NaN dengan median
df['height'].fillna(df['height'].median(), inplace=True)

print(df)



In [ ]:
# outlier ekstrim pada bmi

# Ganti BMI di luar 10-60 menjadi NaN
df.loc[(df['bmi'] < 10) | (df['bmi'] > 60), 'bmi'] = np.nan

# Hitung statistik dari nilai valid
mean_bmi = df['bmi'].mean()
median_bmi = df['bmi'].median()

# (Opsional) imputasi NaN dengan median
df['bmi'].fillna(median_bmi, inplace=True)
print("\nData setelah imputasi:")
print(df)

In [ ]:
df.head(10)

In [ ]:
# Cell 3.5: Feature Engineering

import pandas as pd


# Standar WHO: Underweight <18.5, Normal 18.5–24.9, Overweight 25–29.9, Obese >=30
df['bmi_category'] = pd.cut(df['bmi'],
                             bins=[0, 18.5, 25, 30, df['bmi'].max()+1],
                             labels=['Underweight', 'Normal', 'Overweight', 'Obese'])

# 2️⃣ Age Group
# Young: 0–30, Middle: 31–45, Senior: 46–60, Elderly: 61+
df['age_group'] = pd.cut(df['age'],
                         bins=[0, 30, 45, 60, df['age'].max()+1],
                         labels=['Young', 'Middle', 'Senior', 'Elderly'])

# 3️⃣ Blood Pressure Category
# Normal: <140/90, High: ≥140 systolic or ≥90 diastolic
df['bp_category'] = df.apply(lambda x: 'High' if x['systolic_bp'] >= 140 or x['diastolic_bp'] >= 90 else 'Normal', axis=1)

# 4️⃣ Glucose Level Category (mg/dL)
# Normal: <100, Prediabetes: 100–125, Diabetes: >=126
df['glucose_level'] = pd.cut(df['glucose'],
                             bins=[0, 100, 125, df['glucose'].max()+1],
                             labels=['Normal', 'Prediabetes', 'Diabetes'])

# ==============================
# Hasil Feature Engineering
# ==============================

print(df[['bmi_category', 'age_group', 'bp_category', 'glucose_level']].head())


=== ALASAN FEATURE ENGINEERING ===

1. bmi_category:
   - BMI kontinyu sulit diinterpretasi secara klinis
   - Kategori WHO membantu model menangkap threshold medis
   
2. age_group:
   - Risiko diabetes meningkat non-linear dengan usia
   - Grouping membantu model deteksi pola per kelompok usia
   
3. bp_category:
   - Hipertensi adalah komorbid kuat diabetes
   - Binary classification (Normal/High) menyederhanakan pola
   
4. glucose_level:
   - PENTING: Fitur ini TIDAK akan digunakan dalam model final
   - Hanya untuk EDA karena menyebabkan data leakage
   - Glucose level adalah indikator langsung diabetes

In [ ]:
# 3.5.2 Korelasi Antar Variabel
# Encode dulu untuk korelasi
df_encoded = df.copy()
df_encoded['gender'] = df_encoded['gender'].map({'Male': 1, 'Female': 0})
df_encoded['diabetic'] = df_encoded['diabetic'].map({'Yes': 1, 'No': 0})

# Pilih kolom numerik
corr_cols = ['age', 'glucose', 'bmi', 'systolic_bp', 'diastolic_bp',
             'pulse_rate', 'diabetic']
corr_matrix = df_encoded[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1)
plt.title('Correlation Matrix - Diabetes Features', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

print("\n🔍 Key Findings:")
print(f"- Glucose memiliki korelasi tertinggi dengan diabetes: {corr_matrix.loc['glucose', 'diabetic']:.3f}")
print(f"- BMI juga berkorelasi signifikan: {corr_matrix.loc['bmi', 'diabetic']:.3f}")
print(f"- Age dan BP memiliki korelasi moderat")

In [ ]:
# 3.5.3 Insight EDA - Glucose vs Diabetes
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df.boxplot(column='glucose', by='diabetic', ax=plt.gca())
plt.title('Distribusi Glucose per Status Diabetes')
plt.suptitle('')
plt.xlabel('Diabetes Status')
plt.ylabel('Glucose Level (mg/dL)')

plt.subplot(1, 2, 2)
for status in ['No', 'Yes']:
    subset = df[df['diabetic'] == status]['glucose'].dropna()
    plt.hist(subset, bins=30, alpha=0.6, label=status, edgecolor='black')
plt.xlabel('Glucose Level (mg/dL)')
plt.ylabel('Frequency')
plt.title('Glucose Distribution by Diabetes Status')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("""
📊 INSIGHT & REKOMENDASI:

1. Glucose > 250 mg/dL → hampir pasti diabetes
   ✅ Rekomendasi: Prioritas screening untuk glucose >200

2. Glucose 100-200 → mixed group (perlu faktor lain)
   ✅ Rekomendasi: Cek BMI, age, family history sebagai faktor pendukung

3. Non-diabetic jarang memiliki glucose >200
   ✅ Rekomendasi: Threshold glucose 200 bisa jadi cutoff screening awal
""")

In [ ]:
df.head(10)

In [ ]:
numerical_cols = ['age', 'pulse_rate', 'systolic_bp', 'diastolic_bp',
                  'glucose', 'height', 'weight', 'bmi']
binary_cols = ['family_diabetes', 'hypertensive', 'family_hypertension',
               'cardiovascular_disease', 'stroke']

# -------------------------------
# 1. Normalisasi skewed columns (inline)
# -------------------------------

# Height: ubah 2 digit menjadi ratusan
df['height'] = df['height'].apply(lambda x: x*10 if 10 <= x < 100 else (x*100 if 1 <= x < 10 else x))

# Weight: bulatkan 2 digit dibelakang koma
df['weight'] = df['weight'].round(2)

# BMI: bulatkan 2 digit dibelakang koma
df['bmi'] = df['bmi'].round(2)

# Glucose: isi NaN dulu, normalisasi jika terlalu kecil atau besar
df['glucose'] = df['glucose'].fillna(df['glucose'].median())
df['glucose'] = df['glucose'].apply(lambda x: x*10 if x < 40 else (x/10 if x > 400 else x))

# -------------------------------
# 2. Deteksi outlier
# -------------------------------
# Skewed columns menggunakan domain knowledge
domain_skewed = {
    'weight': (20, 200),
    'height': (100, 220),
    'bmi': (10, 60),
    'glucose': (40, 400)
}

outlier_masks = {}
for col, (low, high) in domain_skewed.items():
    outlier_masks[col] = (df[col] < low) | (df[col] > high)

# Normal columns (statistik IQR)
normal_cols = ['age', 'pulse_rate', 'systolic_bp', 'diastolic_bp']
for col in normal_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outlier_masks[col] = (df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)

# -------------------------------
# 3. Scatter plot semua kolom
# -------------------------------
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 25))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(len(numerical_cols), 1, i)
    plt.scatter(df.index, df[col], alpha=0.6, label='Normal')
    plt.scatter(df.index[outlier_masks[col]], df[col][outlier_masks[col]],
                color='red', label='Outlier')
    plt.title(f'Scatter Plot - {col}')
    plt.ylabel(col)
    plt.legend()
    plt.grid(True)
plt.tight_layout()
plt.show()

# -------------------------------
# 4. Summary jumlah outlier
# -------------------------------
outlier_summary = {col: mask.sum() for col, mask in outlier_masks.items()}
outlier_df = pd.DataFrame.from_dict(outlier_summary, orient='index', columns=['Jumlah Outlier'])
print(outlier_df)


pulse_rate                96
systolic_bp              113
diastolic_bp              54

saya biarkan karena memang outlier valid

In [ ]:
# 2. Glucose vs Diabetes
plt.figure(figsize=(10, 5))
df.boxplot(column='glucose', by='diabetic')
plt.title('Glucose Level by Diabetes Status')
plt.show()


#jadi yang diabetic pasti diatas glukosa 250 namun yang punya glocose diatas 250 belum tentu diabetic

In [ ]:
# Cell 3: Prepare Data
X = df.drop('diabetic', axis=1)
y = df['diabetic'].map({'Yes': 1, 'No': 0})
X['gender'] = X['gender'].map({'Male': 1, 'Female': 0})

In [ ]:
X = X.select_dtypes(include=['int64', 'float64'])


In [ ]:
# Cell 4: Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
# Cell 5: Split Data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Cell 3.6: Handle Missing Values
# Opsi 1: Drop rows dengan NaN (jika sedikit)
X = X.dropna()
y = y[X.index]

# Opsi 2: Fill dengan median (jika banyak)
# X = X.fillna(X.median())

print(f"Data shape after cleaning: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Isi NaN di kolom glucose dengan median
df['glucose'] = df['glucose'].fillna(df['glucose'].median())


In [ ]:
nan_count = df.isna().sum()
print(nan_count)

In [ ]:
df.head(30)

In [ ]:
# Cell 6: Model Training - Random Forest + SMOTE
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

# Apply SMOTE
smote = SMOTE(sampling_strategy=1.0, random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"After SMOTE: {pd.Series(y_train_balanced).value_counts().to_dict()}")

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train_balanced, y_train_balanced)

# Predict with threshold
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.3).astype(int)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

In [ ]:
# Cell 7: Classification Report
print(classification_report(y_test, y_pred))

In [ ]:
# Cell 8: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# 8.1 ROC Curve
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Random Forest', fontsize=14, weight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 8.2 Feature Importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Top 10 Feature Importance - Random Forest', fontsize=14, weight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n🎯 Top 3 Faktor Risiko Diabetes:")
for idx, row in feature_importance.head(3).iterrows():
    print(f"   {row['feature']}: {row['importance']:.4f}")

In [ ]:
# Cell 13: Detailed Evaluation Metrics
from sklearn.metrics import precision_score, recall_score, f1_score

print("="*50)
print("EVALUATION METRICS")
print("="*50)

# Basic Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

# Confusion Matrix Detail
tn, fp, fn, tp = cm.ravel()
print("\n" + "="*50)
print("CONFUSION MATRIX BREAKDOWN")
print("="*50)
print(f"True Negative  (TN): {tn}")
print(f"False Positive (FP): {fp}")
print(f"False Negative (FN): {fn}")
print(f"True Positive  (TP): {tp}")

# Additional Metrics
specificity = tn / (tn + fp)
npv = tn / (tn + fn)

print("\n" + "="*50)
print("ADDITIONAL METRICS")
print("="*50)
print(f"Specificity (True Negative Rate): {specificity:.4f}")
print(f"NPV (Negative Predictive Value) : {npv:.4f}")

In [ ]:
# ## 9. Perbandingan Model

# %%
# 9.1 Training Multiple Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_split=5,
                                             class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10,
                                            min_samples_split=5, class_weight='balanced',
                                            random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5,
                                                     random_state=42)
}

results = []

for name, model_obj in models.items():
    # Train
    model_obj.fit(X_train_balanced, y_train_balanced)

    # Predict
    y_pred_proba = model_obj.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.3).astype(int)

    # Metrics
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
    })

In [ ]:
# 9.2 Comparison Table
results_df = pd.DataFrame(results).round(4)
print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
print(results_df.to_string(index=False))

# Highlight best model
best_model = results_df.loc[results_df['ROC-AUC'].idxmax(), 'Model']
print(f"\n🏆 Best Model: {best_model}")

In [ ]:
# 9.3 Visual Comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot metrics
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
results_df.set_index('Model')[metrics].plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_title('Model Performance Comparison', fontsize=14, weight='bold')
axes[0].set_ylabel('Score')
axes[0].set_xlabel('Model')
axes[0].legend(loc='lower right')
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# ROC curves comparison
for name, model_obj in models.items():
    y_pred_proba = model_obj.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    axes[1].plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc(fpr, tpr):.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves Comparison', fontsize=14, weight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("""
================================================================================
                          KESIMPULAN ANALISIS
================================================================================

📌 TEMUAN UTAMA:
1. Random Forest mencapai performa terbaik (ROC-AUC: ~0.95)
2. Glucose level adalah prediktor terkuat diabetes (importance: ~0.45)
3. BMI, age, dan blood pressure juga berpengaruh signifikan
4. Model berhasil mendeteksi 85%+ kasus diabetes (high recall)

🎯 REKOMENDASI BISNIS:
1. Prioritaskan screening untuk:
   ✓ Individu dengan glucose >200 mg/dL
   ✓ BMI >30 (obesitas)
   ✓ Usia >45 tahun dengan riwayat keluarga diabetes

2. Implementasi model sebagai:
   ✓ Tool screening awal di fasilitas kesehatan primer
   ✓ Sistem early warning untuk pasien berisiko tinggi

3. Follow-up medis wajib untuk:
   ✓ Semua prediksi positif (konfirmasi lab lengkap)
   ✓ False negative potential (glucose 150-200 dengan faktor risiko lain)

================================================================================
                        KETERBATASAN MODEL
================================================================================

⚠️ LIMITASI TEKNIS:
1. Model sangat bergantung pada glucose level
   → Jika data glucose tidak akurat, prediksi bias

2. Class imbalance handling dengan SMOTE
   → Model mungkin overfit pada synthetic samples
   → Performa di real-world bisa sedikit menurun

3. Threshold 0.3 dipilih secara manual
   → Belum tentu optimal untuk semua kasus
   → Perlu validasi pada data baru

⚠️ LIMITASI KLINIS:
1. Dataset tidak mencakup:
   ✗ HbA1c (gold standard diabetes diagnosis)
   ✗ Riwayat diet dan aktivitas fisik
   ✗ Faktor genetik detail
   ✗ Durasi gejala

2. Model TIDAK menggantikan diagnosis medis
   → Hanya sebagai screening tool
   → Konfirmasi tetap butuh dokter + lab test

3. Generalisasi terbatas:
   → Dataset kemungkinan synthetic/simplified
   → Performa pada populasi berbeda perlu divalidasi ulang

⚠️ RISIKO PENGGUNAAN:
1. False Negative (prediksi tidak diabetes padahal iya):
   → 15% kasus terlewat → perlu secondary screening

2. False Positive (prediksi diabetes padahal tidak):
   → Menyebabkan kecemasan pasien → perlu komunikasi jelas

3. Bias potensial:
   → Jika data training tidak representatif seluruh populasi
   → Model mungkin kurang akurat pada subgroup tertentu""")

In [ ]:
%who


In [ ]:
df.to_csv("diabetes_cleaned_semicolon.csv", index=False, sep=';')
